# Tugas
Jobsheet 6 - ANN (Approximate Nearest Neighbors)

Jalankan code berikut, bandingkan hasilnya, dan tuliskan analisa Anda. Tuliskan juga spesifikasi perangkat yang digunakan.

> **Catatan:** Modul meminta menjalankan pada **Google Colab** dan **PyDroid3 (Android)**. Notebook ini dijalankan pada mesin lokal (lihat output di bawah). Silakan jalankan ulang di Colab/PyDroid3 dan lengkapi tabel perbandingan di bagian analisis.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'  # hindari konflik OpenMP antara sklearn, faiss, dan hnswlib

import numpy as np
import time
from sklearn.neighbors import NearestNeighbors
from annoy import AnnoyIndex
import hnswlib
import faiss
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Contoh dataset kecil untuk testing
# -------------------------------
np.random.seed(42)
n_samples = 10000   # jumlah database vector
d = 128             # dimensi
X = np.random.random((n_samples, d)).astype('float32')

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

In [2]:
# -------------------------------
# Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 1.334 s


In [3]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.474 s


In [4]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=d)
p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 3.039 s


In [5]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(d)
index_faiss = faiss.IndexIVFFlat(quantizer, d, 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
_, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.246 s


In [6]:
# -------------------------------
# Tampilkan ringkasan waktu
# -------------------------------
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {time_annoy:.3f}")
print(f"HNSW     : {time_hnsw:.3f}")
print(f"FAISS    : {time_faiss:.3f}")


=== Ringkasan Waktu (detik) ===
Exact NN : 1.334
Annoy    : 0.474
HNSW     : 3.039
FAISS    : 0.246


### Tabel Perbandingan

In [7]:
import pandas as pd

def recall_at_k(pred, gt, k):
    correct = sum(len(set(map(int, pred[i][:k])) & set(map(int, gt[i][:k]))) for i in range(len(pred)))
    return correct / (len(pred) * k)

tabel_tugas = pd.DataFrame([
    {'Metode': 'Exact NN', 'Waktu (s)': round(time_exact, 3), 'Recall@10': 1.0},
    {'Metode': 'Annoy', 'Waktu (s)': round(time_annoy, 3), 'Recall@10': round(recall_at_k(idx_annoy, idx_exact, k), 4)},
    {'Metode': 'HNSW', 'Waktu (s)': round(time_hnsw, 3), 'Recall@10': round(recall_at_k(idx_hnsw, idx_exact, k), 4)},
    {'Metode': 'FAISS IVF', 'Waktu (s)': round(time_faiss, 3), 'Recall@10': round(recall_at_k(idx_faiss, idx_exact, k), 4)},
])
tabel_tugas

,Metode,Waktu (s),Recall@10
0,Exact NN,1.334,1.0000
1,Annoy,0.474,0.1681
2,HNSW,3.039,0.9805
3,FAISS IVF,0.246,0.4182


### Analisis
**Ringkasan hasil (dijalankan pada laptop):**

* **Exact NN** — baseline paling akurat (recall 1.0) tetapi menghitung jarak ke seluruh 10.000 vektor untuk setiap query.
* **Annoy** — membangun 10 pohon proyeksi acak. Waktu query sangat cepat, tetapi karena `search_k` default relatif kecil untuk data 128 dimensi, recall@10 pada percobaan ini rendah (~0,17). Recall dapat dinaikkan dengan memperbesar `search_k`.
* **HNSW** — graf hierarkis dengan `ef=200`; memberi recall paling tinggi (~0,98) di antara metode ANN dan tetap cepat.
* **FAISS IVF** — membagi ruang menjadi 100 klaster dan mencari pada 10 klaster terdekat (`nprobe=10`); recall ~0,42, dapat dinaikkan dengan memperbesar `nprobe`.

**Faktor yang memengaruhi perbedaan hasil antar perangkat:**

1. **CPU & jumlah core.** Annoy dan HNSW memakai multithreading saat build; lebih banyak core → build lebih cepat.
2. **RAM.** Dataset 10.000 × 128 cukup kecil, tetapi di perangkat Android kelas menengah jumlah memori lebih terbatas.
3. **Implementasi library.** `faiss` dan `hnswlib` memakai kode C++ teroptimasi (dan SIMD). Di PyDroid3, ketersediaan wheel/paket dan dukungan native sering menjadi kendala, sehingga waktu bisa jauh lebih lambat atau library tidak tersedia.
4. **Beban latar belakang.** Di smartphone, aplikasi lain dan thermal throttling membuat waktu eksekusi lebih bervariasi.

**Kesimpulan:**

Untuk dataset kecil (10.000 × 128), **exact brute-force masih kompetitif** karena biayanya belum terlalu besar. Keunggulan ANN baru terasa ketika jumlah data sangat besar (jutaan). Di antara metode ANN, **HNSW** umumnya memberi akurasi terbaik dengan kecepatan tinggi, **FAISS IVF** fleksibel dan efisien untuk skala besar, sedangkan **Annoy** paling ringan dan mudah dijalankan bahkan di perangkat terbatas.

> **Spesifikasi perangkat (isi sendiri):**
>
> | Perangkat | OS | CPU | RAM |
> |---|---|---|---|
> | Laptop | macOS | (isi) | (isi) |
> | Smartphone Android | Android | (isi) | (isi) |

**Perbandingan lintas perangkat:**

| Metode | Laptop (detik) | Colab (detik) | PyDroid3 (detik) |
|---|---|---|---|
| Exact NN | (dari output) | (isi) | (isi) |
| Annoy | (dari output) | (isi) | (isi) |
| HNSW | (dari output) | (isi) | (isi) |
| FAISS | (dari output) | (isi) | (isi) |

**Catatan:** PyDroid3 sering tidak menyediakan paket `faiss`/`hnswlib`/`annoy` karena keterbatasan build native di Android. Jika paket tidak tersedia, cukup jalankan metode yang tersedia dan bandingkan.